# 🎬 الحي 404 — تجربة تحويل صورة لفيديو بنموذج Wan 2.2 (مفتوح المصدر)

**من الموبايل، من غير اشتراك مدفوع.** الدفتر يشغل نسخة Wan 2.2 TI2V-5B مع ComfyUI على GPU متاحة في Google Colab، إن توفرت مجانًا. **مش مضمون** إن GPU تكون متاحة أو تقدر تكمل التحميل والتوليد.

**النموذج قوي نسبيًا مقارنةً بالموارد المطلوبة:** ComfyUI الرسمي بيوضح إنه ممكن يشتغل بذاكرة GPU حوالي 8GB مع auto-offloading. ملفات النموذج المطلوبة حجمها حوالي **18GB**، وده تحميل كبير على بيئة Colab.

**مهم:**
- الدفتر لا يُشغل خادمًا عامًا ولا نفقًا/Proxy؛ ComfyUI تعمل محليًا داخل جلسة Colab فقط. وده يتوافق مع إن Colab المجاني مش معمول لاستضافة واجهة ويب خارجية.
- النتيجة **لقطة تجريبية قصيرة صامتة** من الشخصية المرجعية؛ لا صوت مصري أو lip-sync هنا، ولا تعتبر حلقة نهائية.
- جرّب ليلى في لقطة `sh05` أولًا. ما تعملش اعتماد إنتاج كامل قبل مراجعة الملامح والحركة.
- نزّل حزمة الصور `404_mobile_reference_assets.zip` من محادثة ChatGPT، وهي اللي فيها الصور الأصلية الموقّعة SHA-256.
- افتح الدفتر في Chrome من الموبايل، واختار **Runtime → Change runtime type → GPU** لو متاحة، وشغّل كل خلية بزر ▶ بالترتيب.

مصادر: https://docs.comfy.org/tutorials/video/wan/wan2_2 و https://research.google.com/colaboratory/faq.html


In [ ]:
# 1) التأكد من GPU والمساحة (قبل تنزيل أي ملفات كبيرة)
import os, sys, shutil, subprocess
from pathlib import Path
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU مش متاحة في الجلسة. جرّب Runtime → Change runtime type → GPU لو فيه خيار مجاني.")
print("نوع GPU:", torch.cuda.get_device_name(0))
disk=shutil.disk_usage("/content")
print("المساحة الحرة (GB):", round(disk.free/2**30,1))
if disk.free < 24*2**30:
    raise RuntimeError("المساحة الحرة أقل من 24GB؛ لا هنبدأ تحميل ~18GB من ملفات Wan حتى نتجنب امتلاء الجلسة.")
print("✅ عندنا GPU ومساحة مبدئية. ده مش ضمان إن النموذج هيكمل التوليد.")


In [ ]:
# 2) تثبيت ComfyUI وأدوات 404 — لا يفتح سيرفر عام
from pathlib import Path
ROOT=Path("/content/404")
COMFY=Path("/content/ComfyUI")
if not (ROOT/".git").is_dir():
    subprocess.run(["git","clone","--depth","1","https://github.com/omarnet0109-jpg/404.git",str(ROOT)],check=True)
if not (COMFY/".git").is_dir():
    subprocess.run(["git","clone","--depth","1","https://github.com/comfyanonymous/ComfyUI.git",str(COMFY)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(COMFY/"requirements.txt")],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","huggingface_hub","requests"],check=True)
print("✅ ComfyUI وملفات مشروع 404 جاهزين. الخطوة القادمة تحميل ملفات نموذج كبيرة.")


In [ ]:
# 3) تنزيل ملفات النموذج من مستودع Comfy-Org الرسمي (قد يكون تنزيلًا كبيرًا)
from huggingface_hub import hf_hub_download
MODEL_REPO="Comfy-Org/Wan-2.2"
WEIGHTS=Path("/content/404-model-weights")
models=[
    ("diffusion_models","wan2.2_ti2v_5B_fp16.safetensors"),
    ("text_encoders","umt5_xxl_fp8_e4m3fn_scaled.safetensors"),
    ("vae","wan2.2_vae.safetensors"),
]
for folder, name in models:
    remote="split_files/"+folder+"/"+name
    print("تحميل:",name,flush=True)
    source=Path(hf_hub_download(repo_id=MODEL_REPO,filename=remote,local_dir=str(WEIGHTS)))
    if source.stat().st_size < 1024**2:
        raise RuntimeError("ملف النموذج الذي تم تنزيله صغير بشكل غير طبيعي: "+name)
    dest=COMFY/"models"/folder/name
    dest.parent.mkdir(parents=True,exist_ok=True)
    if dest.is_symlink() and not dest.exists():
        dest.unlink()
    if not dest.exists():
        dest.symlink_to(source)
    print("✓", name,round(dest.stat().st_size/2**30,2),"GB")
print("✅ ملفات Wan 2.2 الثلاثة موجودة.")


In [ ]:
# 4) ارفع ملف صور الشخصيات الأصلي ZIP من موبايلك
import json, hashlib, zipfile, io
from google.colab import files
uploaded=files.upload()
if len(uploaded)!=1:
    raise RuntimeError("اختار ملف ZIP واحد فقط: 404_mobile_reference_assets.zip")
filename,data=next(iter(uploaded.items()))
if not filename.lower().endswith(".zip"):
    raise RuntimeError("الملف المطلوب ZIP المرجع الأصلي.")
bible=json.loads((ROOT/"characters"/"character_bible.json").read_text(encoding="utf-8"))
REFS=Path("/content/references")
REFS.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(data)) as z:
    for item in bible["reference_images"]:
        name=item["repo_target"].split("/")[-1]
        raw=z.read("references/"+name)
        actual=hashlib.sha256(raw).hexdigest()
        if actual!=item["sha256"]:
            raise RuntimeError("الصورة لا تطابق مرجع الشخصية المعتمد: "+item["id"])
        (REFS/name).write_bytes(raw)
print("✅ تم اعتماد الصور الأربعة الأصلية بدون تغيير الوجوه.")


In [ ]:
#@title 5) اختار الشخصية لتجربة لقطة صامتة
person = "ليلى" #@param ["ليلى", "وداد", "صابر", "الحارة"]
options={
    "ليلى":("LAYLA","layla.png","sh05"),
    "وداد":("WIDAD","widad.png","sh10"),
    "صابر":("SABER","saber.png","sh04"),
    "الحارة":("ensemble","ensemble.png","sh01"),
}
actor,refname,shot=options[person]
ref=REFS/refname
print("لقطة التجربة:",shot,"الشخصية:",person)
subprocess.run([sys.executable,str(ROOT/"production/free_wan22.py"),
    "--reference",str(ref),"--actor",actor,"--shot",shot,"--frames","33",
    "--width","352","--height","640"],check=True)
print("✅ اختبار تجهيز بدون استخدام GPU. الخطوة الجاية تبدأ التوليد الحقيقي.")


In [ ]:
# 6) تشغيل ComfyUI داخل جلسة Colab فقط (localhost) بدون رابط عام
import time, requests
log_path=Path("/content/comfyui_404.log")
if "comfy_proc" not in globals() or comfy_proc.poll() is not None:
    comfy_log=log_path.open("a")
    comfy_proc=subprocess.Popen([
        sys.executable,"main.py",
        "--listen","127.0.0.1","--port","8188","--lowvram",
        "--disable-auto-launch",
    ],cwd=str(COMFY),stdout=comfy_log,stderr=subprocess.STDOUT)
    print("بدء تحميل نموذج ComfyUI وتشغيله...")
ok=False
for _ in range(90):
    if comfy_proc.poll() is not None:
        raise RuntimeError("ComfyUI لم تبدأ. راجع آخر أسطر في /content/comfyui_404.log")
    try:
        r=requests.get("http://127.0.0.1:8188/object_info",timeout=8)
        if r.ok:
            nodes=r.json()
            required={"Wan22ImageToVideoLatent","SaveVideo","CreateVideo","CLIPLoader","UNETLoader"}
            missing=required-set(nodes)
            if missing:
                raise RuntimeError("ComfyUI قديمة أو تنقصها عقد: "+str(sorted(missing)))
            ok=True
            break
    except requests.RequestException:
        pass
    time.sleep(3)
if not ok:
    print(log_path.read_text(encoding="utf-8",errors="replace")[-2500:])
    raise RuntimeError("تعذّر تشغيل ComfyUI محليًا داخل الجلسة.")
print("✅ ComfyUI شغالة محليًا. مفيش سيرفر مفتوح للإنترنت.")


In [ ]:
#@title 7) توليد فيديو تجريبي حقيقي ثم تنزيله للموبايل
بدء_التوليد = False #@param {type:"boolean"}
if not بدء_التوليد:
    print("علّم على «بدء التوليد» وشغّل الخلية لو مستعد تستهلك وقت GPU المجاني المتاح.")
else:
    result=Path("/content/404_wan22_"+shot+"_preview.mp4")
    cmd=[
        sys.executable,str(ROOT/"production/free_wan22.py"),
        "--reference",str(ref),"--actor",actor,"--shot",shot,
        "--frames","33","--width","352","--height","640",
        "--server","http://127.0.0.1:8188",
        "--output",str(result),"--execute",
    ]
    subprocess.run(cmd,check=True,timeout=1500)
    choices=[result.with_suffix(x) for x in (".mp4",".webm",".mov")]
    completed=next((x for x in choices if x.exists() and x.stat().st_size>1000),None)
    if completed is None:
        raise RuntimeError("مفيش فيديو حقيقي اتولد؛ مش هنعرض معاينة صامتة بدل النتيجة.")
    from IPython.display import Video,display
    if completed.suffix==".mp4":
        display(Video(filename=str(completed),embed=True))
    print("الملف النهائي للتجربة:",completed)
    files.download(str(completed))


## النتيجة والخطوة التالية

لو التجربة نجحت، هيكون عندك فيديو قصير صامت بصيغة MP4/WebM/MOV من صورة الشخصية الأصلية، **مش** فيديو حلقة كاملة. ممكن تراجع الملامح والحركة على الموبايل ثم نوسّع التشغيل. لو Colab فصلت الجلسة أو امتلأت GPU/الذاكرة، ده حد الموارد المجانية وليس خللًا يستوجب الدفع.

**ممنوع اعتبار المقاطع صالحة للنشر قبل مراجعة ثبات الشخصية والصوت المصري والليب-سينك، لأن Wan 2.2 نفسه لا ينتج الحوار.**

بعد التنزيل تقدر ترجع لتطبيق [404 Studio](https://omarnet0109-jpg.github.io/404/mobile-studio/) لإدارة المشاهد. ربط تشغيل Colab كخادم مباشر لواجهة الموبايل **غير مفعّل** لأن سياسة Colab المجانية تقيّد الواجهات الخارجية.
